# Aufgabe 2d: Schnellere Konvergenz als $1/\sqrt{N}$ zur Berechnung von $\pi$

Dieses Jupyter Notebook vergleicht vier verschiedene Verfahren zur Monte-Carlo-Integration eines Viertelkreises im Einheitsquadrat $[0, 1]^2$:

1. **Standard-Monte-Carlo (PRNG, Pseudo-Zufall):** Konvergenz $\mathcal{O}(N^{-0{,}50}) = 1/\sqrt{N}$
2. **Stratified Sampling (Schichtstichprobe im Gitter):** Konvergenz $\mathcal{O}(N^{-0{,}75}) = 1/N^{3/4}$
3. **Quasi-Monte-Carlo mit Sobol-Folge (Low-Discrepancy):** Konvergenz $\mathcal{O}(N^{-0{,}75} \dots N^{-1{,}0})$
4. **Quasi-Monte-Carlo mit Halton-Folge (Low-Discrepancy):** Konvergenz $\mathcal{O}(N^{-0{,}75} \dots N^{-1{,}0})$

Durch die Direktive `%matplotlib inline` werden alle Plots **direkt im Notebook** gerendert (keine externen Pop-up-Fenster). Zusätzlich werden alle Grafiken als eigenständige `.png`-Dateien abgespeichert.

In [ ]:
# Inline-Plotting aktivieren (keine externen Pop-ups)
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import qmc

# Plot-Stil für übersichtliche Darstellung
plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.size'] = 11

---
## 1. Standard Pseudo-Random Monte Carlo (PRNG)

### Mathematischer Hintergrund:
Gegeben ist das Einheitsquadrat $[0, 1]^2$. Ein Punkt $(x, y)$ liegt im Viertelkreis, wenn:
$$x^2 + y^2 \le 1{,}0$$
Punkte auf dem Rand zählen laut Aufgabenstellung als **innerhalb**.

Bei Verwendung von stochastisch unabhängigen Zufallszahlen (i.i.d.) ist jeder gezogene Punkt ein Bernoulli-Experiment mit Erfolgswahrscheinlichkeit $p = \frac{\pi}{4}$. Nach dem **Zentralen Grenzwertsatz (CLT)** skaliert die Standardabweichung des Schätzers mit:
$$\sigma_{\hat{\pi}} = \sqrt{\frac{\pi(4-\pi)}{N}} \propto \frac{1}{\sqrt{N}} = \mathcal{O}(N^{-0{,}5})$$
Reiner Zufall führt zwangsläufig zu lokaler Klumpenbildung und Leerstellen.

In [ ]:
def mc_standard(N: int, seed: int = 42) -> float:
    """
    Standard-Monte-Carlo-Verfahren unter Verwendung von Pseudo-Zufallszahlen (PRNG).
    
    FUNKTIONSWEISE:
    1. Erzeugt N unabhängige, identisch gleichverteilte Zufallspunkte (x_i, y_i) in [0, 1]^2.
    2. Prüft für jeden Punkt die Viertelkreis-Bedingung: x^2 + y^2 <= 1.0 (Rand inklusive).
    3. Berechnet den Schätzwert: pi_est = 4.0 * sum(inside) / N.
    
    Parameter:
    ----------
    N : int
        Anzahl der zu generierenden Zufallspunkte.
    seed : int, optional
        Seed für Reproduzierbarkeit (Standard: 42).
        
    Rückgabe:
    ---------
    float
        Geschätzter Wert für Pi.
    """
    rng = np.random.default_rng(seed)
    x = rng.uniform(0.0, 1.0, size=N)
    y = rng.uniform(0.0, 1.0, size=N)
    
    inside = (x**2 + y**2) <= 1.0
    num_inside = np.sum(inside)
    
    return 4.0 * num_inside / N

---
## 2. Stratified Sampling (Schichtstichprobenverfahren)

### Mathematischer Hintergrund:
Das Einheitsquadrat wird in ein reguläres Gitter aus $M \times M$ Zellen ($N = M^2$) zerlegt. In jeder Zelle wird genau ein Punkt platziert (mit gleichverteiltem Zufalls-Jitter):
$$x = \frac{i + u_x}{M}, \quad y = \frac{j + u_y}{M} \quad \text{mit } u_x, u_y \in [0, 1)$$

- Zellen, die komplett im Viertelkreis liegen, liefern exakt $1$ (Varianz $= 0$).
- Zellen, die komplett außerhalb liegen, liefern exakt $0$ (Varianz $= 0$).
- **Varianz entsteht nur an den Grenzzellen**, die vom Kreisbogen geschnitten werden.

Die Anzahl der Grenzzellen entlang des Bogens ist $\mathcal{O}(M) = \mathcal{O}(\sqrt{N})$. Die Gesamtvarianz sinkt auf $\mathcal{O}(\sqrt{N} \cdot N^{-2}) = \mathcal{O}(N^{-3/2})$. Damit konvergiert der Fehler mit:
$$\sigma = \sqrt{\text{Var}} = \mathcal{O}(N^{-3/4}) = \mathcal{O}(N^{-0{,}75})$$
Dies ist asymptotisch deutlich schneller als Standard-Monte-Carlo!

In [ ]:
def mc_stratified(N_target: int, seed: int = 42) -> tuple[int, float]:
    """
    Stratified Sampling (Schichtstichprobenverfahren / Jittered Grid Sampling).
    
    FUNKTIONSWEISE:
    1. Unterteilt [0, 1]^2 in M x M Gitterzellen (M = round(sqrt(N_target)), N = M^2).
    2. Platziert in jeder Zelle genau einen Zufallspunkt mit Versatz u in [0, 1) / M.
    3. Prüft x^2 + y^2 <= 1.0 (Rand inklusive).
    4. Berechnet pi_est = 4.0 * sum(inside) / N.
    
    Parameter:
    ----------
    N_target : int
        Angestrebte Punktezahl (wird auf die nächste Quadratzahl M^2 gerundet).
    seed : int, optional
        Seed für die Zufallsgenerierung (Standard: 42).
        
    Rückgabe:
    ---------
    tuple[int, float]
        (Tatsächliche Punktanzahl N, Geschätzter Wert für Pi).
    """
    rng = np.random.default_rng(seed)
    M = int(np.round(np.sqrt(N_target)))
    N = M * M
    
    i = np.arange(M)
    j = np.arange(M)
    ii, jj = np.meshgrid(i, j)
    
    u_x = rng.uniform(0.0, 1.0, size=(M, M))
    u_y = rng.uniform(0.0, 1.0, size=(M, M))
    
    x = (ii + u_x) / M
    y = (jj + u_y) / M
    
    inside = (x**2 + y**2) <= 1.0
    pi_est = 4.0 * np.sum(inside) / N
    
    return N, pi_est

---
## 3. Quasi-Monte-Carlo: Sobol-Folge

### Mathematischer Hintergrund:
Die **Sobol-Folge** ist eine deterministische Niedrigdiskrepanz-Folge (*Low-Discrepancy Sequence*). Sie basiert auf binärer radikaler Inversion und Richtungsvektoren, sodass binäre Teilintervalle (dyadische Boxen) optimal homogen belegt werden.

Für zweidimensionale Mengen mit glattem Rand erreichen Niedrigdiskrepanz-Folgen theoretisch und empirisch Konvergenzraten zwischen $\mathcal{O}(N^{-0{,}75})$ und $\mathcal{O}(N^{-1{,}0})$.

In [ ]:
def mc_sobol(m: int, scramble: bool = False) -> tuple[int, float]:
    """
    Quasi-Monte-Carlo-Integration (QMC) mit der Sobol-Folge.
    
    FUNKTIONSWEISE:
    1. Verwendet scipy.stats.qmc.Sobol zur Generierung von N = 2^m Punkten in [0, 1]^2.
    2. Prüft x^2 + y^2 <= 1.0 (Rand inklusive).
    3. Berechnet pi_est = 4.0 * sum(inside) / N.
    
    Parameter:
    ----------
    m : int
        Zweierpotenz-Exponent (Punktanzahl N = 2^m).
    scramble : bool, optional
        Standard: False (rein deterministische Folge).
        
    Rückgabe:
    ---------
    tuple[int, float]
        (Punktanzahl N = 2^m, Geschätzter Wert für Pi).
    """
    N = 2**m
    sampler = qmc.Sobol(d=2, scramble=scramble, seed=42 if scramble else None)
    pts = sampler.random(N)
    
    x = pts[:, 0]
    y = pts[:, 1]
    
    inside = (x**2 + y**2) <= 1.0
    pi_est = 4.0 * np.sum(inside) / N
    
    return N, pi_est

---
## 4. Quasi-Monte-Carlo: Halton-Folge

### Mathematischer Hintergrund:
Die **Halton-Folge** verallgemeinert die 1D-Van-der-Corput-Folge. In zwei Dimensionen werden zwei unterschiedliche Primzahlen als Basen verwendet (Basis 2 für die $x$-Achse und Basis 3 für die $y$-Achse). Durch radikale Inversion werden Punkte deterministisch so platziert, dass Klumpenbildung und Lücken vollständig vermieden werden.

In [ ]:
def mc_halton(N: int) -> float:
    """
    Quasi-Monte-Carlo-Integration (QMC) mit der Halton-Folge.
    
    FUNKTIONSWEISE:
    1. Verwendet scipy.stats.qmc.Halton zur Erzeugung von N Punkten in [0, 1]^2
       (Basen: 2 für x, 3 für y).
    2. Prüft x^2 + y^2 <= 1.0 (Rand inklusive).
    3. Berechnet pi_est = 4.0 * sum(inside) / N.
    
    Parameter:
    ----------
    N : int
        Anzahl der Punkte.
        
    Rückgabe:
    ---------
    float
        Geschätzter Wert für Pi.
    """
    sampler = qmc.Halton(d=2, scramble=False)
    pts = sampler.random(N)
    
    x = pts[:, 0]
    y = pts[:, 1]
    
    inside = (x**2 + y**2) <= 1.0
    return 4.0 * np.sum(inside) / N

---
## 5. Visueller Vergleich der Punktverteilungsmuster

Hier visualisieren wir direkt im Notebook, wie die vier Verfahren den Raum im Einheitsquadrat abdecken.

In [ ]:
def visualize_sampling_patterns(N_sample: int = 576):
    """
    Erstellt einen visuellen 2x2-Vergleich der vier Punktverteilungsmuster im Einheitsquadrat.
    Speichert das Bild als 'sampling_patterns.png' und zeigt es direkt inline an.
    """
    fig, axes = plt.subplots(2, 2, figsize=(10, 10), dpi=130)
    fig.suptitle(f"Vergleich der Sampling-Muster im Einheitsquadrat (N = {N_sample})", fontsize=15, fontweight="bold")
    
    theta = np.linspace(0, np.pi / 2, 200)
    arc_x = np.cos(theta)
    arc_y = np.sin(theta)
    
    # 1. Standard Pseudo-Random
    rng = np.random.default_rng(42)
    x_prng = rng.uniform(0, 1, N_sample)
    y_prng = rng.uniform(0, 1, N_sample)
    
    # 2. Stratified Sampling
    M = int(np.round(np.sqrt(N_sample)))
    i = np.arange(M)
    j = np.arange(M)
    ii, jj = np.meshgrid(i, j)
    x_strat = ((ii + rng.uniform(0, 1, (M, M))) / M).flatten()
    y_strat = ((jj + rng.uniform(0, 1, (M, M))) / M).flatten()
    
    # 3. Sobol Sequence (auf nächste Zweierpotenz 512 runden)
    m_sobol = int(np.round(np.log2(N_sample)))
    pts_sobol = qmc.Sobol(d=2, scramble=False).random(2**m_sobol)
    x_sobol, y_sobol = pts_sobol[:, 0], pts_sobol[:, 1]
    
    # 4. Halton Sequence
    pts_halton = qmc.Halton(d=2, scramble=False).random(N_sample)
    x_halton, y_halton = pts_halton[:, 0], pts_halton[:, 1]
    
    methods = [
        ("Standard Pseudo-Random (PRNG)\n(Zufällige Klumpen & Leerstellen)", x_prng, y_prng, False),
        ("Stratified Grid Sampling\n(1 Jitter-Punkt pro Zelle)", x_strat, y_strat, True),
        (f"Quasi-Monte-Carlo: Sobol\n(Niedrigdiskrepanz, N = {2**m_sobol})", x_sobol, y_sobol, False),
        ("Quasi-Monte-Carlo: Halton\n(Niedrigdiskrepanz, Basen 2 und 3)", x_halton, y_halton, False)
    ]
    
    for ax, (title, x_pts, y_pts, draw_grid) in zip(axes.flatten(), methods):
        inside = (x_pts**2 + y_pts**2) <= 1.0
        
        ax.scatter(x_pts[inside], y_pts[inside], s=12, color="#1f77b4", alpha=0.75, label="Innen")
        ax.scatter(x_pts[~inside], y_pts[~inside], s=12, color="#d62728", alpha=0.75, label="Außen")
        ax.plot(arc_x, arc_y, "k-", linewidth=2.5, label=r"$x^2 + y^2 = 1$")
        
        if draw_grid:
            grid_lines = np.linspace(0, 1, M + 1)
            for gl in grid_lines:
                ax.axvline(gl, color="gray", linestyle=":", linewidth=0.5, alpha=0.5)
                ax.axhline(gl, color="gray", linestyle=":", linewidth=0.5, alpha=0.5)
                
        ax.set_xlim(-0.02, 1.02)
        ax.set_ylim(-0.02, 1.02)
        ax.set_aspect("equal")
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.legend(loc="lower left", fontsize=8)
        
    plt.tight_layout()
    pattern_file = "sampling_patterns.png"
    plt.savefig(pattern_file)
    print(f"Sampling-Muster-Plot gespeichert als: {pattern_file}")
    plt.show()

# Sampling-Muster direkt ausführen und inline anzeigen
visualize_sampling_patterns(N_sample=576)

---
## 6. Numerische Verifikation der Konvergenzraten

Wir berechnen nun den Fehler $|\hat{\pi} - \pi|$ für $N = 2^6$ bis $2^{20}$ (64 bis 1.048.576 Punkte).
Für stochastische Methoden wird über 15 unabhängige Durchläufe gemittelt.
Anschließend ermitteln wir die empirische Konvergenzrate $\alpha$ über lineare Regression:
$$\ln(\text{Fehler}) = -\alpha \cdot \ln(N) + \ln(C) \iff \text{Fehler} \propto N^{-\alpha}$$

In [ ]:
def fit_convergence_rate(N_values: list[int], errors: list[float]) -> float:
    """
    Ermittelt die empirische Konvergenzrate alpha aus einem Datensatz.
    Lineare Regression auf: ln(Fehler) = -alpha * ln(N) + ln(C).
    """
    n_arr = np.array(N_values, dtype=float)
    err_arr = np.array(errors, dtype=float)
    
    valid = err_arr > 0.0
    log_n = np.log(n_arr[valid])
    log_err = np.log(err_arr[valid])
    
    slope, _ = np.polyfit(log_n, log_err, 1)
    return -slope

def verify_faster_convergence():
    """
    Führt die vollständige numerische Untersuchung der Konvergenzraten durch,
    speichert 'faster_convergence.png' und zeigt den Plot inline an.
    """
    m_values = list(range(6, 21))
    N_values = [2**m for m in m_values]
    
    mc_std_errors = []
    strat_errors = []
    sobol_errors = []
    halton_errors = []
    
    num_trials = 15  # Mittelung über unabhängige Trials für stochastische Verfahren
    
    print("=" * 76)
    print("  VERGLEICH DER KONVERGENZRATEN ZUR BERECHNUNG VON PI (Aufgabe 2d)")
    print("=" * 76)
    print(f"{'N':>10} | {'Std-MC (PRNG)':>14} | {'Stratified':>14} | {'Sobol (QMC)':>14} | {'Halton (QMC)':>14}")
    print("-" * 76)
    
    for m, N in zip(m_values, N_values):
        # 1. Standard-MC (Mittelwert über num_trials Läufe)
        std_errs = [abs(mc_standard(N, seed=seed) - np.pi) for seed in range(num_trials)]
        avg_std_err = float(np.mean(std_errs))
        mc_std_errors.append(avg_std_err)
        
        # 2. Stratified Sampling (Mittelwert über num_trials Läufe)
        strat_errs = []
        for seed in range(num_trials):
            actual_N, p_strat = mc_stratified(N, seed=seed)
            strat_errs.append(abs(p_strat - np.pi))
        avg_strat_err = float(np.mean(strat_errs))
        strat_errors.append(avg_strat_err)
        
        # 3. Sobol-Folge
        _, p_sobol = mc_sobol(m, scramble=False)
        err_sobol = abs(p_sobol - np.pi)
        sobol_errors.append(err_sobol)
        
        # 4. Halton-Folge
        p_halton = mc_halton(N)
        err_halton = abs(p_halton - np.pi)
        halton_errors.append(err_halton)
        
        print(f"{N:>10,d} | {avg_std_err:>14.6e} | {avg_strat_err:>14.6e} | {err_sobol:>14.6e} | {err_halton:>14.6e}")
        
    # Gefittete Konvergenzraten
    alpha_mc = fit_convergence_rate(N_values, mc_std_errors)
    alpha_strat = fit_convergence_rate(N_values, strat_errors)
    alpha_sobol = fit_convergence_rate(N_values, sobol_errors)
    alpha_halton = fit_convergence_rate(N_values, halton_errors)
    
    print("-" * 76)
    print("EMPIRISCHE KONVERGENZRATEN (Steigung im Log-Log-Plot: Fehler ~ N^-alpha):")
    print(f"  Standard Pseudo-Random MC:  alpha = {alpha_mc:.2f}  (Theorie: 0.50 = 1/sqrt(N))")
    print(f"  Stratified Sampling:        alpha = {alpha_strat:.2f}  (Theorie: 0.75 = 1/N^(3/4))")
    print(f"  Quasi-Monte-Carlo (Sobol):  alpha = {alpha_sobol:.2f}  (Theorie: ~0.75 bis 1.0)")
    print(f"  Quasi-Monte-Carlo (Halton): alpha = {alpha_halton:.2f}  (Theorie: ~0.75 bis 1.0)")
    print("=" * 76)
    
    # Plot erstellen
    plt.figure(figsize=(10, 6.5), dpi=150)
    plt.loglog(N_values, mc_std_errors, 'o-', color='#1f77b4', label=f'Standard MC (fit: $N^{{-{alpha_mc:.2f}}}$)', markersize=5)
    plt.loglog(N_values, strat_errors, 's-', color='#ff7f0e', label=f'Stratified Sampling (fit: $N^{{-{alpha_strat:.2f}}}$)', markersize=5)
    plt.loglog(N_values, sobol_errors, '^-', color='#2ca02c', label=f'QMC Sobol (fit: $N^{{-{alpha_sobol:.2f}}}$)', markersize=5)
    plt.loglog(N_values, halton_errors, 'd-', color='#9467bd', label=f'QMC Halton (fit: $N^{{-{alpha_halton:.2f}}}$)', markersize=5)
    
    ref_N = np.array(N_values, dtype=float)
    plt.loglog(ref_N, 1.6 / np.sqrt(ref_N), 'k--', alpha=0.5, label=r'Theorie-Referenz: $O(N^{-1/2}) = 1/\sqrt{N}$')
    plt.loglog(ref_N, 1.6 / (ref_N**0.75), 'k:', alpha=0.7, label=r'Theorie-Referenz: $O(N^{-3/4}) = 1/N^{3/4}$')
    plt.loglog(ref_N, 2.0 / ref_N, 'k-.', alpha=0.5, label=r'Theorie-Referenz: $O(N^{-1}) = 1/N$')
    
    plt.xlabel('Anzahl der Punkte $N$', fontsize=12)
    plt.ylabel(r'Absoluter Fehler $|\hat{\pi} - \pi|$', fontsize=12)
    plt.title(r'Konvergenzvergleich zur $\pi$-Berechnung: Standard-MC vs. Stratified & QMC', fontsize=13, fontweight='bold')
    plt.grid(True, which="both", ls="--", alpha=0.6)
    plt.legend(fontsize=10, loc='best')
    plt.tight_layout()
    
    conv_file = "faster_convergence.png"
    plt.savefig(conv_file)
    print(f"Konvergenz-Plot gespeichert als: {conv_file}")
    plt.show()

In [ ]:
# Konvergenz-Verifikation ausführen und Plot inline anzeigen
verify_faster_convergence()